## Data Validation Report

This notebook validates the production fraud monitoring dataset
to ensure data integrity and reliability before dashboard usage.

Validation checks:

- Null values
- Negative transaction amounts
- Duplicate transactions
- Risk score validity
- Fraud alert consistency

#### ->Load Production Table

In [0]:
df = spark.table("production_fraud_monitoring")

In [0]:
print("Total records:", df.count())

Total records: 5000000


#### ->Null Check

In [0]:
from pyspark.sql.functions import col

null_check = df.filter(
    col("transfer_id").isNull() |
    col("account_id").isNull() |
    col("amount").isNull()
)

print("Null Records:", null_check.count())

Null Records: 0


#### ->Negative Transaction Check

In [0]:
negative_txn = df.filter(col("amount") < 0)

print("Negative transactions:", negative_txn.count())

Negative transactions: 0


#### ->Duplicate Transaction Check

In [0]:
duplicates = df.groupBy("transfer_id") \
.count() \
.filter(col("count") > 1)

print("Duplicate transactions:", duplicates.count())

Duplicate transactions: 0


#### ->Risk Score Validation

In [0]:
invalid_risk = df.filter(col("risk_score") < 0)

print("Invalid risk scores:", invalid_risk.count())

Invalid risk scores: 0


#### ->Fraud Alert Validation

In [0]:
fraud_flag_check = df.filter(~col("suspicious").isin(0,1))

print("Invalid fraud flags:", fraud_flag_check.count())

Invalid fraud flags: 0


#### ->Validation Summary

In [0]:
validation_results = [
("Null Records", null_check.count()),
("Negative Transactions", negative_txn.count()),
("Duplicate Transactions", duplicates.count()),
("Invalid Risk Scores", invalid_risk.count()),
("Invalid Fraud Flags", fraud_flag_check.count())
]

validation_df = spark.createDataFrame(
validation_results,
["check_name","failed_records"]
)

validation_df.show()

+--------------------+--------------+
|          check_name|failed_records|
+--------------------+--------------+
|        Null Records|             0|
|Negative Transact...|             0|
|Duplicate Transac...|             0|
| Invalid Risk Scores|             0|
| Invalid Fraud Flags|             0|
+--------------------+--------------+



#### ->Save Validation Report

In [0]:
validation_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("validation_fraud_pipeline_report")